# Fama-French Regressions by Year and Decile
Este notebook recalcula as regressões de Fama-French de 3 Fatores para as carteiras (decis) formadas pelas métricas de rede (HRM e Pozzi).
A lógica atual utiliza os ativos agrupados em 10 decis para cada ano, construídos _In-Sample_ (ano $t$), e avaliados _Out-of-Sample_ (ano $t+1$).\n

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)\n

## 1. Processamento e Regressões OLS
Aqui nós iteramos pelos anos (2014 a 2024), carregamos as rentabilidades diárias do ano seguinte, filtramos os tickers de cada decil e rodamos a regressão OLS.
Usamos o cálculo de **Equal-Weighted** log returns para os portfólios, garantindo que a carteira capte o efeito puramente estrutural da rede (sem enviesar por Market Cap).\n

In [ ]:
# Carrega os fatores de Fama-French
factors_df = pd.read_parquet("../../data/02_clean/fama_french_factors.parquet")

# Os fatores originais geralmente vêm em porcentagem (ex: 1.5%), então dividimos por 100
factors_df = factors_df / 100

years = range(2014, 2025)
metrics = ['hrm', 'pozzi']
deciles = [f'decil_{i}' for i in range(1, 11)]

# Dicionário para armazenar resultados puros das regressões
regression_results = []

for metric in metrics:
    print(f"Processando regressões para: {metric.upper()}...")
    
    # Carrega metadados que dizem qual Ticker está em qual decil a cada ano
    try:
        df_meta = pd.read_parquet(f"../../data/07_portfolios_metadata/complete_metadata_{metric}.parquet")
    except FileNotFoundError:
        print(f"Arquivo de metadados para {metric} não encontrado. Pule.")
        continue
    
    for year in years:
        # Puxa retornos Out-of-Sample (ano + 1)
        try:
            oos_ret = pd.read_parquet(f"../../data/02_clean/returns_new_{year+1}.parquet")
        except FileNotFoundError:
            continue
            
        # Alinha os fatores de Fama-French às datas dos retornos daquele ano
        factors_year = factors_df[(factors_df.index >= oos_ret.index[0]) & (factors_df.index <= oos_ret.index[-1])]
        
        for decil in deciles:
            # Puxa tickers daquele decil no ano 'year'
            tickers = df_meta[(df_meta['year'] == year) & (df_meta['portfolio'] == decil)]['Ticker'].tolist()
            
            # Garante que as ações existam na base de retornos do ano t+1
            valid_tickers = [t for t in tickers if t in oos_ret.columns]
            
            if len(valid_tickers) == 0:
                continue
                
            # Calcula retorno Equal-Weighted da carteira
            # Primeiro tiramos log_returns e depois a média diária dos ativos
            port_ret = np.log1p(oos_ret[valid_tickers]).mean(axis=1)
            
            # Subtrai a Risk-Free rate para obter o Excesso de Retorno
            excess_ret = port_ret - factors_year['RF']
            
            # Variáveis independentes
            X = factors_year[['Mkt-RF', 'SMB', 'HML']]
            X = sm.add_constant(X)
            
            # Alinhamento por data (inner join) e drop de NAs
            aligned = pd.concat([excess_ret.rename("ExRet"), X], axis=1, join="inner").dropna()
            
            if len(aligned) < 30: # Evita regressões com poucos dias
                continue
                
            # Roda a Regressão OLS Clássica
            model = sm.OLS(aligned['ExRet'], aligned[['const', 'Mkt-RF', 'SMB', 'HML']]).fit()
            
            regression_results.append({
                'metric': metric,
                'year_oos': year + 1,      # Ano OOS (o que os retornos ocorreram)
                'year_formed': year,       # Ano em que a carteira foi formada
                'decil': decil,
                'alpha': model.params['const'] * 252, # Alpha Anualizado
                'alpha_tstat': model.tvalues['const'],
                'mkt_beta': model.params['Mkt-RF'],
                'smb_beta': model.params['SMB'],
                'hml_beta': model.params['HML'],
                'r_squared': model.rsquared
            })

df_results = pd.DataFrame(regression_results)
print("Todas as regressões foram calculadas!")
\n

## 2. Opção de Tabela: Resumo Estilo Fama-MacBeth
Esta tabela sintetiza 10 anos de regressões. Ela exibe a **média temporal** dos Alphas e dos Betas para cada decil. 
O teste t (Alpha_tstat_FM) é calculado dividindo a média do Alpha pelo Erro Padrão da média (Desvio Padrão do Alpha / raiz do número de anos). Essa é a forma mais clássica de relatar performance em asset pricing.\n

In [ ]:
def generate_fama_macbeth_table(df_metric_results, metric_name):
    if df_metric_results.empty:
        return None
        
    # Número de anos na amostra OOS
    n_years = df_metric_results['year_oos'].nunique()
    
    # Agrega tirando a média das variáveis e o desvio padrão do alpha
    summary = df_metric_results.groupby('decil').agg({
        'alpha': ['mean', 'std'],
        'mkt_beta': 'mean',
        'smb_beta': 'mean',
        'hml_beta': 'mean',
        'r_squared': 'mean'
    })
    
    # Calcula T-Statistic no estilo Fama-MacBeth (Mean / Standard Error)
    summary['Alpha_tstat_FM'] = summary[('alpha', 'mean')] / (summary[('alpha', 'std')] / np.sqrt(n_years))
    
    # Achata as colunas (Flatten)
    summary.columns = ['Alpha', 'Alpha_Std', 'Mkt_Beta', 'SMB_Beta', 'HML_Beta', 'R_Squared', 'Alpha_tstat_FM']
    
    # Adiciona carteira Long-Short
    # Pega valores do Decil 1 (Central) e Decil 10 (Peripheral)
    d1 = summary.loc['decil_1']
    d10 = summary.loc['decil_10']
    
    # Long Central, Short Peripheral (se desejar inverter, mude d1 - d10 para d10 - d1)
    ls_row = d1 - d10
    
    # Ajusta R_squared e t-stat do Long-Short (o t-stat correto exigiria a série long-short ano a ano)
    # Como atalho, calculamos a série histórica da carteira L-S e depois aplicamos o mean/std
    df_ls = df_metric_results.pivot(index='year_oos', columns='decil', values='alpha')
    df_ls['L_S'] = df_ls['decil_1'] - df_ls['decil_10']
    ls_alpha_mean = df_ls['L_S'].mean()
    ls_alpha_std = df_ls['L_S'].std()
    ls_tstat = ls_alpha_mean / (ls_alpha_std / np.sqrt(n_years))
    
    ls_row['Alpha'] = ls_alpha_mean
    ls_row['Alpha_Std'] = ls_alpha_std
    ls_row['Alpha_tstat_FM'] = ls_tstat
    ls_row['R_Squared'] = np.nan # R2 não se subtrai
    
    summary.loc['Central - Peripheral (L-S)'] = ls_row
    
    # Reordena para ficar bonito (decil_1 até decil_10 e depois L-S)
    decil_order = [f'decil_{i}' for i in range(1, 11)] + ['Central - Peripheral (L-S)']
    summary = summary.reindex(decil_order)
    
    # Formata para visualização
    final_table = summary[['Alpha', 'Alpha_tstat_FM', 'Mkt_Beta', 'SMB_Beta', 'HML_Beta', 'R_Squared']].copy()
    final_table = final_table.round(4)
    
    # Renomeia o índice
    index_names = [f'Decil {i} (Central)' if i==1 else f'Decil {i} (Peripheral)' if i==10 else f'Decil {i}' for i in range(1, 11)]
    index_names.append('Long-Short (Central - Periph)')
    final_table.index = index_names
    
    print(f"\n=== FAMA-MACBETH REGRESSION SUMMARY: {metric_name.upper()} ===")
    display(final_table)
    
    return final_table

# Executa para HRM e Pozzi
fm_hrm = generate_fama_macbeth_table(df_results[df_results['metric'] == 'hrm'], 'hrm')
fm_pozzi = generate_fama_macbeth_table(df_results[df_results['metric'] == 'pozzi'], 'pozzi')

# Salva tabelas
if fm_hrm is not None:
    fm_hrm.to_csv("../../data/07_portfolios_metadata/table_famamacbeth_hrm.csv")
if fm_pozzi is not None:
    fm_pozzi.to_csv("../../data/07_portfolios_metadata/table_famamacbeth_pozzi.csv")
\n

## 3. Opção de Gráfico: Heatmap de Alphas por Ano e Decil
O gráfico mostra de forma clara onde e quando a sua estratégia apresentou *Alpha* fora da curva ao longo de todo o período, quebrando o paradigma de que só importam médias gerais.\n

In [ ]:
def plot_alpha_heatmap(df_metric_results, metric_name, save_dir="../../figures"):
    if df_metric_results.empty:
        return
        
    # Prepara dados (Linhas: Decil, Colunas: Ano)
    pivot_alpha = df_metric_results.pivot(index='decil', columns='year_oos', values='alpha')
    
    # Ordena os decis
    pivot_alpha.index = pd.Categorical(pivot_alpha.index, categories=[f"decil_{i}" for i in range(1, 11)], ordered=True)
    pivot_alpha = pivot_alpha.sort_index()
    
    # Melhora o nome do eixo Y
    y_labels = [f"Decil {i}" for i in range(1, 11)]
    y_labels[0] = "Decil 1 (Central)"
    y_labels[-1] = "Decil 10 (Peripheral)"
    
    plt.figure(figsize=(12, 6))
    
    # Plot do Heatmap
    ax = sns.heatmap(
        pivot_alpha, 
        annot=True,        # Mostra o valor de Alpha na célula
        fmt=".3f",         # 3 casas decimais
        cmap="coolwarm_r", # Vermelho negativo, Azul positivo
        center=0,          # 0 fica branco/neutro
        linewidths=0.5, 
        linecolor='white',
        annot_kws={"size": 11}
    )
    
    ax.set_yticklabels(y_labels, rotation=0, fontsize=12)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=12)
    
    plt.title(f"Annualized Alpha by Decile and Year — {metric_name.upper()}", fontsize=18, fontweight="bold", pad=15)
    plt.xlabel("Year (Out-of-Sample)", fontsize=14, labelpad=10)
    plt.ylabel("Portfolio", fontsize=14, labelpad=10)
    
    plt.tight_layout()
    
    # Salva
    Path(save_dir).mkdir(parents=True, exist_ok=True)
    plt.savefig(f"{save_dir}/heatmap_alpha_{metric_name}.png", dpi=300, bbox_inches="tight")
    plt.show()

# Plota para ambos
plot_alpha_heatmap(df_results[df_results['metric'] == 'hrm'], 'hrm')
plot_alpha_heatmap(df_results[df_results['metric'] == 'pozzi'], 'pozzi')
\n